# MEC 2026 - Online Music Alignment with Matchmaker (Symbolic)

**Music Alignment Workshop, MEC 2026 — Section 3 (Online Alignment), Part 2 (Symbolic)**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pymatchmaker/mec2026_alignment_workshop/blob/online-symbolic-alignment/notebooks/online_alignment_symbolic.ipynb)

This notebook accompanies the online alignment section of **Music Alignment Uncovered: Representations, Algorithms and Hands-On Tools** at MEC 2026.

In this notebook we align **one MIDI recording and the corresponding score** using the [Matchmaker](https://github.com/pymatchmaker/matchmaker) package. 

## 0. Setup

Run the setup cells once at the beginning. If you already installed the dependencies, you may skip the installation cells.

Workshop resources live under `mec-variation/` at the repo root. Make sure that directory is present alongside this notebook (clone the workshop repo if you haven't).

For audio score following, Matchmaker may need FluidSynth and libsndfile. In Colab the apt cell below installs them; locally install them via your system package manager, e.g. `conda install -c conda-forge fluidsynth libsndfile`.

In [ ]:
#@title Setup dependencies (apt + pip + clone) { display-mode: "form" }
import importlib.util
import shutil
import subprocess
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB and shutil.which("apt-get") is not None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "fluidsynth", "libsndfile1"],
        check=True,
    )
else:
    print("Skipping apt-get setup outside Colab.")

%pip install -q "setuptools>=80,<81" "numpy>=1.26.3,<2.0" git+https://github.com/pymatchmaker/matchmaker.git@develop pandas matplotlib librosa partitura ipython

if IN_COLAB and not Path("/content/mec2026_alignment_workshop").exists():
    !git clone --depth 1 --branch online-symbolic-alignment https://github.com/pymatchmaker/mec2026_alignment_workshop.git

if IN_COLAB:
    print("\nRestarting runtime so the downgraded numpy is loaded cleanly...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
#@title Imports and data paths { display-mode: "form" }
import json
import warnings
import librosa
import matchmaker as mm
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import partitura as pt
from IPython.display import Audio, Image, display, clear_output
from pathlib import Path
from queue import Empty
from matchmaker import Matchmaker
from matchmaker.features.audio import ChromagramProcessor, CQTProcessor, CQTSpectralFluxProcessor
from matchmaker.features.midi import onset_pianoroll, PitchClassPianoRollProcessor, ChordOnsetProcessor, CHORD_THRESHOLD
from matchmaker.matchmaker import AVAILABLE_METHODS, DEFAULT_METHOD, DEFAULT_PROCESSOR
from matchmaker.io.stream import STREAM_END
from matchmaker.io.audio import QUEUE_TIMEOUT
from partitura.musicanalysis.performance_codec import get_time_maps_from_alignment

warnings.filterwarnings("ignore", category=np.exceptions.ComplexWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="partitura")
warnings.filterwarnings("ignore", category=UserWarning, module="hiddenmarkov")
print("matchmaker", mm.__version__)

DATA_DIR = Path("../mec-variation")
if not DATA_DIR.exists():
    DATA_DIR = Path("/content/mec2026_alignment_workshop/mec-variation")
SCORE_FILE = DATA_DIR / "scores/musicxml/short.musicxml"
AUDIO_FILE = DATA_DIR / "performances/wav/short1.wav"
MIDI_FILE = DATA_DIR / "performances/midi/short1.mid"
MATCH_FILE = DATA_DIR / "alignments/match/short1.match"
SNIPPET_FILE = DATA_DIR / "performances/wav/snippet.wav"
MIDI_SNIPPET_FILE = DATA_DIR / "performances/midi/snippet.mid"
SNIPPET_SCORE_FILE = DATA_DIR / "scores/musicxml/snippet.musicxml"
PREVIEW_IMAGE = DATA_DIR / "scores/simple_mozart_first_two_measures.png"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

perf, alignment, score = pt.load_match(filename=str(MATCH_FILE), create_score=True)
snote_array = score.note_array()
_, stime_to_ptime_map = get_time_maps_from_alignment(
    perf.note_array(), snote_array, alignment, onset_aggregation_fun=np.min,
)
NOTE_ANNOTATIONS = stime_to_ptime_map(np.unique(snote_array["onset_beat"]))

print("Data directory:", DATA_DIR.resolve())
for path in [SCORE_FILE, AUDIO_FILE, MIDI_FILE, MATCH_FILE, SNIPPET_FILE, PREVIEW_IMAGE]:
    print(path.name, "ok" if path.exists() else "missing")
print(f"NOTE_ANNOTATIONS: {len(NOTE_ANNOTATIONS)} entries, {NOTE_ANNOTATIONS.min():.3f}s ~ {NOTE_ANNOTATIONS.max():.3f}s")

## 1. MIDI vs. Audio Online Alignment

### Input Representation:

- **Audio**
  - Derived from acoustic signals: 
    1. Chroma, 
    2. CQT + Spectral Flux, etc.

- **MIDI**
  - Derived from symbolic note events such as pitch, onset, offset, velocity: 
    1. Pitch + Onset, 
    2. Pitch Class + Onset,  
    3. Pianoroll, etc.

---


#### Advantages of MIDI
- No processing is required to decipher the musical note being played.
- Precise note onset and offset timing.
- Faster and cleaner feature extraction and matching --> lower latency.
- Robust to noise, timbre, and recording conditions.

#### Disadvantages of MIDI
- At best, provides approximated expressive acoustic information (timbre, articulation, dynamics).
- MIDI is most naturally suited to piano/keyboard-style/percussion note events;
  but not to instruments such as the violin or the human voice, 
  which would depend on the transcription of their audio signals into MIDI information.
- In comparison to audio recordings, fewer MIDI recordings of performances are available.

---

## 2. MIDI Features

The `MidiStream` in Matchmaker is responsible for sending the input MIDI data (either in windowed frames, or by note event) through a `Processor` that returns one feature vector.

Let's look at two processors in this segment of the workshop:

- `PitchClassPianoRollProcessor` — Equivalent to a 12-bin chroma representation in audio processing. Both the input MIDI and the reference score features are reduced to 12 pitch-class values. This processor is used by the Particle Filter method.
- `ChordOnsetProcessor` — Processes incoming note events by setting the corresponding pitch indices in the feature vector to 1 whenever a note is played, and remains so until it has stopped playing. If multiple note events start to play within a 10 ms window, it produces a multi-hot vector representing all detected pitches. Depending on the selected pitch range, the processor outputs either an 88-bin vector (piano range) or a full 128-bin vector. This processor is used by the Outer Product HMM method and the OLTW Arzt method.

#### Let's first look again at the score snippet that we are working with, and listen to the MIDI snippet of the same

In [ ]:
print(f"Below is the short score snippet again, along with the MIDI snippet (first 2 measures).")
display(Image(filename=str(PREVIEW_IMAGE), width=700))
perf = pt.load_performance_midi(str(MIDI_SNIPPET_FILE))
perf_audio = pt.utils.synthesize(perf)
sr = pt.utils.globals.SAMPLE_RATE
display(Audio(perf_audio, rate=sr))

# Cached once: score image + its aspect ratio (used for figure sizing below).
SCORE_IMG = mpimg.imread(str(PREVIEW_IMAGE))
SCORE_AR = SCORE_IMG.shape[1] / SCORE_IMG.shape[0]

## Now, let's visualize the output of these processors as a comparison to the audio processors that we have already seen.
#### First let's select the score follower that we would like to work with:

## Select Score Follower from the dropdown menu in the cell below this one:

**Particle Filter:**
In this notebook, the Particle Filter uses the PitchClassPianoRollProcessor

**Outer Product HMM:**
The Outer Product HMM uses the ChordOnsetProcessor

**OLTW Arzt:**
The Online Time-Warping Arzt method also uses the ChordOnsetProcessor

In [ ]:
from ipywidgets import Dropdown
from IPython.display import display

method = 'outerhmm'  # Default method

# Interactive model selection
model_dropdown = Dropdown(
    options={'Particle Filter': 'pf', 'Outer Product HMM': 'outerhmm', 'OLTW Arzt': 'arzt'},
    value='outerhmm',
    description='Model:'
)

# Handle selection
def on_model_change(change):
    global method
    method = change['new']
    print(f"Selected method: {method}")

model_dropdown.observe(on_model_change, names='value')
display(model_dropdown)

In [ ]:
'''Uncomment the line below to manually set the method without using the dropdown.'''
#method = 'outerhmm' # 'pf', 'arzt'

matchmaker = mm.Matchmaker(
    score_file=SCORE_FILE if method != 'pf' else SNIPPET_SCORE_FILE,
    performance_file=MIDI_FILE if method != 'pf' else MIDI_SNIPPET_FILE,
    input_type="midi",
    method=method,
)

sf = matchmaker.score_follower

if method == 'outerhmm':
    pianoroll = np.zeros((88, 20))
    frame_skip = 3
elif method == 'pf':
    pianoroll = np.zeros((12, 20))
    frame_skip = 5
elif method == 'arzt':
    pianoroll = np.zeros((88, 20))
    frame_skip = 3

pianoroll_cursor_end_point = pianoroll.shape[1] * 3 // 4
pianoroll_cursor_start_point = pianoroll.shape[1] * 2 // 5
def update_pianoroll(pianoroll, feature):
    pianoroll[:, :-1] = pianoroll[:, 1:]
    pianoroll[:, -1] = feature

    return pianoroll

fig_y = 8
fig_x = 16

fig_text = 5 if method == 'pf' else 45
post_stream = 15

fig, ax = plt.subplots(ncols=1, figsize=(fig_x, fig_y))
count = 1



def plot_features(pianoroll, pianoroll_cursor_end_point, pianoroll_cursor_start_point, fig_text, fig, ax, count):
    ax.clear()

    ax.imshow(pianoroll, aspect='auto', origin='lower')
    ax.set_title("Method: '{}'. Input Frame: {}".format(method, count))
    ax.set_xticks([])
            # add a vertical line at the cursor point
    ax.axvline(x=pianoroll_cursor_end_point+0.5, color='r', linestyle='--')
    ax.axvline(x=pianoroll_cursor_start_point-2.5, color='g', linestyle='--')
            # colour the background of the plot to the right of the cursor point till the end of the image in grey
    ax.axvspan(pianoroll_cursor_end_point+0.5, pianoroll.shape[1], color='purple')
    ax.axvspan(0, pianoroll_cursor_start_point-2.5, color='green')
    ax.set_xlim(0, pianoroll.shape[1])
            # add text to the right of the cursor point saying "MIDI Stream Processor"
    ax.text(pianoroll_cursor_end_point+1, fig_text, "<- MIDI Processor", fontsize=16, color='white')
    ax.text(1, fig_text, "Score Follower <--", fontsize=16)
    ax.set_xlabel("Input Time (frames)")
    ax.set_ylabel("Pitch")
    ax.grid(False, axis='y')

    clear_output(wait=True)
    display(fig)

flag=False
with matchmaker.stream:
    while not flag:
        features = sf.queue.get(timeout=QUEUE_TIMEOUT)
        if features is STREAM_END:
            print(f"End of stream reached with {count}. Plotting remaining frames...")
            for _ in range(post_stream):
                pianoroll = update_pianoroll(pianoroll, np.zeros_like(pianoroll.shape[0]))
                if count % frame_skip == 0 or count == 1:
                    plot_features(pianoroll, pianoroll_cursor_end_point, pianoroll_cursor_start_point, fig_text, fig, ax, count)
                count += 1
            break
        else:
            beat = sf(features[0], features[1])
            pianoroll = update_pianoroll(pianoroll, features[0])

        if count % frame_skip == 0 or count == 1:
            plot_features(pianoroll, pianoroll_cursor_end_point, pianoroll_cursor_start_point, fig_text, fig, ax, count)                

        count += 1

ax.clear()
clear_output(wait=True)
plt.close()



## 3. Matchmaker Quick Run Alignment Recap, now with MIDI:

```python
from matchmaker import Matchmaker

mm = Matchmaker(
    score_file="path/to/score.musicxml",
    performance_file="path/to/performance.mid",
    input_type="midi",
    method="outerhmm",
)

for current_position in mm.run():
    print(current_position)
```

The yielded value is the estimated score position in beats. After a run, `mm.score_follower.alignment_path` stores a `(2, T)` array:

- row 0: estimated score beat
- row 1: performance time in seconds

We will use the helper functions below so that each practical example returns the same kind of result dictionary.

### 3.1 Reading Matchmaker's score positions

Matchmaker reports the current position in **score beats**. For this Mozart example, the score positions come from Partitura's `note_array()` representation, especially the `onset_beat` field.

The unique note-onset beats become the main discrete score states used by several online followers.

In [ ]:
score = pt.load_musicxml(str(SCORE_FILE))
score_part = score[0]
note_array = score_part.note_array()
score_positions = np.unique(note_array["onset_beat"])

print(f"Number of notes: {len(note_array)}")
print(f"Number of unique score onset positions: {len(score_positions)}")
print("First 8 score positions in beats (first measure):")
print(score_positions[:8])

display(Image(filename=str(PREVIEW_IMAGE), width=700))

pd.DataFrame(note_array[:10])[["id", "pitch", "onset_beat", "duration_beat"]]

In [ ]:
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Results will be saved in {RESULTS_DIR.resolve()}")


## 5. MIDI score following

MIDI methods available in Matchmaker include:

- `"pthmm"`: pitch-based HMM score follower. This is the default for MIDI.
- `"outerhmm"`: outer-product HMM for symbolic input.
- `"arzt"` and `"dixon"`: event-based online time warping variants.
- `"pf"`:  particle filter for probabilistic score position and tempo estimation.

The default MIDI processor is the `ChordOnsetProcessor`, which groups near-simultaneous note events into chord observations.

In [ ]:
performance = pt.load_performance_midi(str(MIDI_FILE))
performance_notes = performance.note_array()
perf_pitches = performance_notes["pitch"]

print("Number of performed MIDI notes:", len(performance_notes))
print("Performance note-array fields:", performance_notes.dtype.names)

midi_pianoroll, midi_onsets = onset_pianoroll(
    performance_notes,
    onset_key="onset_sec",
    piano_range=True,
)

score_pianoroll, score_onsets = onset_pianoroll(
    score_part.note_array(),
    onset_key="onset_beat",
    piano_range=True,
)

unique_score_onsets = np.unique(score_onsets)

fig, ax = plt.subplots(figsize=(20, 8))
img = ax.imshow(midi_pianoroll.T, origin="lower", aspect="auto", interpolation="nearest")
ax.set_xlabel("Chord/event index")
ax.set_ylabel("Piano key index (A0-C8)")
ax.set_title("MIDI onset piano-roll features")
fig.colorbar(img, ax=ax, label="Active pitch")
plt.show()

pd.DataFrame(performance_notes[:10])[["pitch", "onset_sec", "duration_sec"]]

### 5.1 PitchHMM

`pthmm` is the default MIDI method. It uses pitch observations and a probabilistic state model to estimate which score onset is currently active.

In [ ]:
print(f"Performance file: {MIDI_FILE.name}, with input mode: midi")
print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'pthmm'...")

midi_pthmm_mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=MIDI_FILE,
    input_type="midi",
    method="pthmm",
)

fig, ax = plt.subplots(figsize=(10, 6))
midi_pthmm_positions = []
i = 0
for current_position in midi_pthmm_mm.run(verbose=False):
    ax.clear()
    ax.plot(current_position, 0, "ro")
    #ax.annotate(f"Beat: {current_position:.2f}", xy=(current_position, 0), textcoords="offset points", xytext=(current_position, 2.5), ha='center', fontsize=12)
    ax.set_yticks([])
    ax.set_xlabel("Score Position (beats)")
    ax.set_title("Real-time Estimated Score Position")
    ax.set_xlim(0, score_positions.max()+1)
    clear_output(wait=True)
    display(fig)
    midi_pthmm_positions.append(float(current_position))
    i += 1

ax.clear()
plt.close()

midi_pthmm_path = midi_pthmm_mm.score_follower.alignment_path

print("MIDI observations:", len(midi_pthmm_positions))
print("Alignment path shape:", midi_pthmm_path.shape)
print("First 10 estimated beat positions:")
print(np.round(midi_pthmm_positions[:10], 3))

midi_pthmm_eval = midi_pthmm_mm.run_evaluation(
    perf_annotations=NOTE_ANNOTATIONS,
    debug=True,
    save_dir=RESULTS_DIR,
    run_name="midi_pthmm",
    level="note",
    plot_dist_matrix=False,
)

print("Evaluation result:")
print(json.dumps(midi_pthmm_eval, indent=2))
display(Image(filename=str(RESULTS_DIR / "midi_pthmm.png"), width=700))

### 5.2 OuterHMM

Now run the symbolic OuterHMM separately. Keeping this plot separate from PitchHMM makes it easier to inspect the path shape and any failures.

In [ ]:
midi_outerhmm_mm = None
midi_outerhmm_positions = []
midi_outerhmm_path = None
midi_outerhmm_eval = None

try:
    print(f"Performance file: {MIDI_FILE.name}, with input mode: midi")
    print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'outerhmm'...")

    midi_outerhmm_mm = Matchmaker(
        score_file=SCORE_FILE,
        performance_file=MIDI_FILE,
        input_type="midi",
        method="outerhmm",
    )

    fig, ax = plt.subplots(figsize=(10, 2))
    for current_position in midi_outerhmm_mm.run(verbose=False):
        ax.clear()
        ax.plot(current_position, 0, "ro")
        ax.set_yticks([])
        ax.set_xlabel("Score Position (beats)")
        ax.set_title("Real-time Estimated Score Position")
        ax.set_xlim(0, score_positions.max() + 1)
        clear_output(wait=True)
        display(fig)
        midi_outerhmm_positions.append(float(current_position))

    ax.clear()
    plt.close()

    midi_outerhmm_path = midi_outerhmm_mm.score_follower.alignment_path

    print("MIDI OuterHMM observations:", len(midi_outerhmm_positions))
    print("Alignment path shape:", midi_outerhmm_path.shape)
    print("First 10 estimated beat positions:")
    print(np.round(midi_outerhmm_positions[:10], 3))

    midi_outerhmm_eval = midi_outerhmm_mm.run_evaluation(
        perf_annotations=NOTE_ANNOTATIONS,
        debug=True,
        save_dir=RESULTS_DIR,
        run_name="midi_outerhmm",
        level="note",
        plot_dist_matrix=False,
    )

    print("Evaluation result:")
    print(json.dumps(midi_outerhmm_eval, indent=2))
    display(Image(filename=str(RESULTS_DIR / "midi_outerhmm.png"), width=700))
except Exception as exc:
    print("MIDI OuterHMM demo skipped:")
    print(type(exc).__name__, exc)

### 5.3 Particle Filter

A Particle Filter approaches the alignment task slightly differently. It tracks many hypothetical beat/tempo states (“particles”) as the performance input arrives, where each particle predicts where the next beat or score position should occur.
As new notes arrive, particles that better match the input receive higher weights, while poor matches are discarded or resampled into stronger positions.
Over time, the particle cloud converges on the most likely alignment between the performed audio and the reference score.

In [ ]:
pf_mm = None
pf_positions = []
pf_path = None
pf_eval = None

try:
    print(f"Performance file: {MIDI_FILE.name}, with input mode: midi")
    print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'pf'...")

    pf_mm = Matchmaker(
        score_file=SCORE_FILE,
        performance_file=MIDI_FILE,
        input_type="midi",
        method="pf",
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    count = 0
    for current_position in pf_mm.run(verbose=False):
        if count % 20 == 0 or count == 1:
            ax.clear()
            ax.plot(pf_mm.score_follower.x, np.zeros(1000), "ro")
            # plot the tempo mean as green crosses along with its value on the y-axis
            ax.plot(current_position, pf_mm.score_follower.tempo_mean, "gx", markersize=10, label="Estimated Tempo")
            ax.annotate(f"{pf_mm.score_follower.tempo_mean:.2f} BPM", (current_position, pf_mm.score_follower.tempo_mean), textcoords="offset points", xytext=(40,-5), ha='center', color='green')
            ax.plot(current_position, 0, "bx", markersize=10, label="Estimated Position")
            ax.annotate(f"Beat: {current_position:.2f}", (current_position, 0), textcoords="offset points", xytext=(0,10), ha='center', color='blue')
            ax.set_ylim(-10, pf_mm.score_follower.v_max)
            ax.set_xlabel("Score Position (beats)")
            ax.set_title("Real-time Estimated Score Position")
            ax.set_xlim(0, score_positions.max() + 1)
            ax.legend()
            clear_output(wait=True)
            display(fig)
        pf_positions.append(float(current_position))
        count += 1

    ax.clear()
    plt.close()

    pf_path = pf_mm.score_follower.alignment_path

    print("Particle Filter observations:", len(pf_positions))
    print("Alignment path shape:", pf_path.shape)
    print("First 10 estimated beat positions:")
    print(np.round(pf_positions[:10], 3))

    pf_eval = pf_mm.run_evaluation(
        perf_annotations=NOTE_ANNOTATIONS,
        debug=True,
        save_dir=RESULTS_DIR,
        run_name="pf",
        level="note",
        plot_dist_matrix=False,
    )

    print("Evaluation result:")
    print(json.dumps(pf_eval, indent=2))
    display(Image(filename=str(RESULTS_DIR / "pf.png"), width=700))

except Exception as exc:
    print("Particle Filter demo skipped:")
    print(type(exc).__name__, exc)

## 6. MIDI evaluation

Compare the MIDI PitchHMM and MIDI OuterHMM evaluation metrics in one table.

In [ ]:
midi_eval_items = [
    ("Pitch HMM", len(midi_pthmm_positions), midi_pthmm_eval),
]

if midi_outerhmm_eval is not None:
    midi_eval_items.append(
        ("Outer Product HMM", len(midi_outerhmm_positions), midi_outerhmm_eval)
    )

if pf_eval is not None:
    midi_eval_items.append(
        ("Particle Filter", len(pf_positions), pf_eval)
    )

midi_eval_rows = []
for method, observations, evaluation in midi_eval_items:
    row = {"method": method, "observations": observations}
    for key, value in evaluation.items():
        if isinstance(value, dict):
            for subkey, subvalue in value.items():
                row[f"{key}.{subkey}"] = subvalue
        else:
            row[key] = value
    midi_eval_rows.append(row)

pd.DataFrame(midi_eval_rows)

## 7. Practical variations

Small changes to try during the workshop:

Audio:

- Change the audio method from `"arzt"` to `"dixon"`.
- Change the audio processor from `"chroma"` to `"mfcc"`, `"cqt"`, or `"mel"`.
- Inspect the saved files in `results/`: alignment path TSV, ground-truth TSV, JSON metrics, and debug plot.

MIDI:

- Try MIDI `"hmm"` or `"arzt"` in the playground cell and inspect each plot separately.
- Look for places where symbolic methods react differently to event grouping or pitch errors.

The important thing to watch is not only whether a final alignment looks correct, but how quickly and stably the online estimate moves while the performance unfolds.

In [ ]:
# Example playground cell.
# Edit these values and rerun.

PLAY_INPUT_TYPE = "midi"   # "audio" or "midi"
PLAY_METHOD = "dixon"       # audio: "arzt", "dixon", "outerhmm"; midi: "pthmm", "hmm", "outerhmm", "arzt"
PLAY_PROCESSOR = None        # e.g. "chroma", "mfcc", "cqt", "mel" for audio

performance_file = AUDIO_FILE if PLAY_INPUT_TYPE == "audio" else MIDI_FILE
run_name = f"playground_{PLAY_INPUT_TYPE}_{PLAY_METHOD}"

try:
    playground_mm = Matchmaker(
        score_file=SCORE_FILE,
        performance_file=performance_file,
        input_type=PLAY_INPUT_TYPE,
        method=PLAY_METHOD,
        processor=PLAY_PROCESSOR,
    )
    playground_positions = []
    for current_position in playground_mm.run(verbose=False):
        playground_positions.append(float(current_position))

    playground_eval = playground_mm.run_evaluation(
        perf_annotations=NOTE_ANNOTATIONS,
        debug=True,
        save_dir=RESULTS_DIR,
        run_name=run_name,
        level="note",
        plot_dist_matrix=False,
    )

    print("Tracked observations:", len(playground_positions))
    print("Evaluation result:")
    print(json.dumps(playground_eval, indent=2))
    display(Image(filename=str(RESULTS_DIR / f"{run_name}.png"), width=700))
except Exception as exc:
    print("Playground run failed:")
    print(type(exc).__name__, exc)